# CircuitSynth: End-to-End Implementation and Vietnamese Evaluation

Notebook này trình bày toàn bộ thực nghiệm của đồ án CircuitSynth trên WebNLG, DART, ZebraLogic và phần mở rộng tiếng Việt trên ViSFD. Mã nguồn chính vẫn nằm trong các module `.py`; notebook dùng trực tiếp source và artifacts của repository để kiểm tra pipeline, phân tích dữ liệu, tổng hợp kết quả, so sánh baseline/ablation, phân tích lỗi và demo tiếng Việt.

Cấu trúc repository được sử dụng:

```text
CIRCUITSYNTH/
├── src/                         # pipeline chính
├── artifacts/                   # artifacts English
├── run.py
├── config.yaml
├── tests/
├── CircuitSynth_ViSFD/
│   ├── src/                     # source ViSFD riêng
│   ├── artifacts/               # artifacts ViSFD
│   ├── run.py
│   ├── config.yaml
│   └── tests/
└── notebooks/
    └── CircuitSynth_Project.ipynb
```


## 0. Environment setup

Notebook tự tìm project root trên máy local, Google Colab hoặc Kaggle. Không sử dụng đường dẫn tuyệt đối của máy cá nhân.


In [1]:
from pathlib import Path
import json
import os
import sys
import subprocess
import importlib.util
from collections import Counter


def is_main_root(path: Path) -> bool:
    return (
        (path / "run.py").is_file()
        and (path / "config.yaml").is_file()
        and (path / "src").is_dir()
        and (path / "CircuitSynth_ViSFD").is_dir()
    )


def find_main_root() -> Path:
    cwd = Path.cwd().resolve()
    direct = [cwd, *cwd.parents, Path("/content"), Path("/kaggle/working")]
    for path in direct:
        if path.exists() and is_main_root(path):
            return path.resolve()

    for base in (Path("/content"), Path("/kaggle/working"), cwd):
        if not base.exists():
            continue
        for child in base.rglob("CircuitSynth_ViSFD"):
            root = child.parent
            if is_main_root(root):
                return root.resolve()

    raise FileNotFoundError(
        "Không tìm thấy project root. Hãy upload hoặc giải nén toàn bộ repository CIRCUITSYNTH "
        "trước khi chạy notebook."
    )


MAIN_ROOT = find_main_root()
VISFD_ROOT = MAIN_ROOT / "CircuitSynth_ViSFD"
EN_ARTIFACTS = MAIN_ROOT / "artifacts"
VI_ARTIFACTS = VISFD_ROOT / "artifacts"

print("Main project :", MAIN_ROOT)
print("Main src     :", MAIN_ROOT / "src")
print("ViSFD project:", VISFD_ROOT)
print("ViSFD src    :", VISFD_ROOT / "src")


Main project : C:\Users\ASUS\Desktop\CircuitSynth
Main src     : C:\Users\ASUS\Desktop\CircuitSynth\src
ViSFD project: C:\Users\ASUS\Desktop\CircuitSynth\CircuitSynth_ViSFD
ViSFD src    : C:\Users\ASUS\Desktop\CircuitSynth\CircuitSynth_ViSFD\src


In [2]:
# Cài các dependency còn thiếu. Source chính và source ViSFD có cùng dependency stack.
required_modules = {
    "yaml": "PyYAML",
    "numpy": "numpy",
    "scipy": "scipy",
    "datasets": "datasets",
    "torch": "torch",
    "transformers": "transformers",
    "peft": "peft",
    "bitsandbytes": "bitsandbytes",
    "pytest": "pytest",
    "pandas": "pandas",
}
missing = [pkg for module, pkg in required_modules.items() if importlib.util.find_spec(module) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{MAIN_ROOT}[full,test]", "pandas"],
        check=True,
    )
else:
    print("Required packages are already available.")


Installing missing packages: ['PyYAML', 'numpy', 'scipy', 'datasets', 'torch', 'transformers', 'peft', 'bitsandbytes', 'pytest', 'pandas']


In [3]:
import pandas as pd
import yaml
import torch

print("Python:", sys.version.split()[0])
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


Python: 3.11.9
CUDA: False
GPU: CPU


## 1. Repository and pipeline checks

Hai pipeline dùng hai thư mục `src` riêng. Các lệnh được chạy với `cwd` tương ứng để tránh import nhầm package giữa pipeline chính và ViSFD.


In [29]:
def run_cmd(args, cwd: Path, check=True):
    env = os.environ.copy()
    env["PYTHONIOENCODING"] = "utf-8"
    env["PYTHONUTF8"] = "1"

    result = subprocess.run(
        args,
        cwd=str(cwd),
        text=True,
        encoding="utf-8",
        errors="replace",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
    )

    print(result.stdout)

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}: {args}"
        )

    return result


for label, root in [("English/Main", MAIN_ROOT), ("ViSFD", VISFD_ROOT)]:
    print(f"\n=== {label} pipeline status ===")
    run_cmd([sys.executable, "run.py", "status"], root)



=== English/Main pipeline status ===
{
  "scientific_config_hash": "43692d8398b414f61c0609ab00cb50a230282c2eba0dc538c0a748ff4080b5fb",
  "stages": [
    {
      "stage": "C0_data_schema",
      "status": "complete",
      "compatible_complete": true,
      "reason": "complete",
      "record_count": 40830,
      "current_step": null
    },
    {
      "stage": "C1_silver",
      "status": "complete",
      "compatible_complete": true,
      "reason": "complete",
      "record_count": 4000,
      "current_step": null
    },
    {
      "stage": "C2_psdd_theta_star",
      "status": "complete",
      "compatible_complete": true,
      "reason": "complete",
      "record_count": 4000,
      "current_step": null
    },
    {
      "stage": "C3_psdd_theta_prime",
      "status": "complete",
      "compatible_complete": true,
      "reason": "complete",
      "record_count": 4000,
      "current_step": 6
    },
    {
      "stage": "C4_sampled_plans",
      "status": "complete",
      "comp

In [5]:
print("=== Main pipeline tests ===")
run_cmd([sys.executable, "-m", "pytest", "-q"], MAIN_ROOT)

print("\n=== ViSFD pipeline tests ===")
run_cmd([sys.executable, "-m", "pytest", "-q"], VISFD_ROOT)


=== Main pipeline tests ===
..................                                                       [100%]


=== ViSFD pipeline tests ===
.......................                                                  [100%]



CompletedProcess(args=['c:\\Users\\ASUS\\AppData\\Local\\Programs\\Python\\Python311\\python.exe', '-m', 'pytest', '-q'], returncode=0, stdout='\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b.\x1b\x1b                                                  [100%]\x1b\n')

## 2. English benchmark data and preprocessing

Pipeline chính sử dụng WebNLG, DART và ZebraLogic. Cell dưới đọc trực tiếp manifest và prepared records đã tạo bởi Stage 1, thay vì tải lại toàn bộ benchmark chỉ để trình bày notebook.


In [6]:
def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def read_jsonl(path: Path):
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def require(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Required artifact is missing: {path}")
    return path


main_manifests = read_json(require(EN_ARTIFACTS / "data" / "manifests.json"))
manifest_df = pd.DataFrame(main_manifests)[
    ["dataset", "official_split", "configured_population_count", "sample_fraction", "selected_count", "duplicate_count", "leakage_count"]
]
manifest_df


,dataset,official_split,configured_population_count,sample_fraction,selected_count,duplicate_count,leakage_count
0,webnlg,test,1779,0.5,890,0,0
1,webnlg,train,13211,0.5,6606,0,0
2,webnlg,validation,1667,0.5,834,0,0
3,dart,test,5000,0.5,2500,0,0
4,dart,train,50000,0.5,25000,0,0
5,zebralogic,easy,3000,0.5,1500,0,0
6,zebralogic,hard,3000,0.5,1500,0,0
7,zebralogic,medium,4000,0.5,2000,0,0


In [7]:
main_records = read_jsonl(require(EN_ARTIFACTS / "data" / "records.jsonl"))

english_split_counts = (
    pd.DataFrame(main_records)
    .groupby(["dataset", "official_split"], dropna=False)
    .size()
    .reset_index(name="records")
)
english_split_counts


,dataset,official_split,records
0,dart,test,2500
1,dart,train,25000
2,webnlg,test,890
3,webnlg,train,6606
4,webnlg,validation,834
5,zebralogic,research_test,500
6,zebralogic,research_train,4000
7,zebralogic,research_validation,500


In [8]:
# Một số prepared records để thấy dữ liệu đầu vào đã được chuyển thành semantic plan như thế nào.
examples = []
for dataset in ["webnlg", "dart", "zebralogic"]:
    row = next(r for r in main_records if r["dataset"] == dataset)
    examples.append({
        "dataset": dataset,
        "split": row["official_split"],
        "source_id": row["source_id"],
        "reference_plan": row["reference_plan"],
        "reference_text": row.get("reference_text", ""),
    })
pd.DataFrame(examples)


,dataset,split,source_id,reference_plan,reference_text
0,webnlg,test,webnlg:test:web_nlg_en-test-1,"{'domain': 'webnlg', 'triples': [{'object': '1...","Nie Haisheng, born on October 13, 1964, worked..."
1,dart,test,dart:test:dart-test-10,"{'domain': 'dart', 'fields': [{'column': 'WINN...",The date of the winning score 74-67-71-73=285 ...
2,zebralogic,research_train,zebralogic:easy:generated-easy-00000,"{'assignment': {'animal': {'dog': 1, 'fish': 0...",There are 2 houses numbered from left to right...


## 3. Main CircuitSynth implementation and training artifacts

Pipeline chính gồm 8 stage: data/schema, Teacher + verifier, semantic prior, distributional projection, plan sampling, Student QLoRA training, FSA-constrained generation và evaluation.


In [9]:
main_config = yaml.safe_load((MAIN_ROOT / "config.yaml").read_text(encoding="utf-8"))
main_train = read_json(require(EN_ARTIFACTS / "student" / "training_report.json"))
main_quota = read_json(require(EN_ARTIFACTS / "silver" / "quota_report.json"))
main_projection = read_json(require(EN_ARTIFACTS / "circuits" / "projection_report.json"))

pd.DataFrame([
    {
        "Teacher": main_config["teacher"]["model"],
        "Student": main_config["student"]["model"],
        "Silver accepted": main_quota["accepted_total"],
        "Student train": main_train["train_count"],
        "Student validation": main_train["validation_count"],
        "Optimizer steps": main_train["executed_optimizer_steps"],
        "Epochs": main_config["student"]["epochs"],
        "QLoRA r": main_config["student"]["lora_rank"],
        "Gradient accumulation": main_config["student"]["gradient_accumulation_steps"],
    }
])


,Teacher,Student,Silver accepted,Student train,Student validation,Optimizer steps,Epochs,QLoRA r,Gradient accumulation
0,Qwen/Qwen2.5-1.5B-Instruct,Qwen/Qwen2.5-0.5B-Instruct,4000,3800,200,238,1,8,16


In [10]:
pd.DataFrame([
    {
        "Projection converged": main_projection["objective_converged"],
        "Iterations": main_projection["iterations"],
        "Optimality residual": main_projection["optimality_residual"],
        "Invalid support mass": main_projection["invalid_support_mass"],
        "Normalized": main_projection["normalized"],
        "Solver": main_projection["solver"],
    }
])


,Projection converged,Iterations,Optimality residual,Invalid support mass,Normalized,Solver
0,True,6,2.902509e-07,0.0,True,convex_dual_lbfgs


## 4. English test evaluation

Các bảng dưới được tính trực tiếp từ `artifacts/metrics/metrics.json` và `artifacts/outputs/generated.jsonl` của lần chạy đã hoàn tất.


In [11]:
main_metrics = read_json(require(EN_ARTIFACTS / "metrics" / "metrics.json"))
main_full = main_metrics["metrics"]["full_pipeline"]

english_full_rows = []
for dataset, block in main_full["by_dataset"].items():
    english_full_rows.append({
        "Dataset": dataset,
        "n": block["output_count"],
        "SV (%)": block["schema_validity_percent"],
        "Factuality": block["factuality_precision"],
        "Fact coverage": block["fact_coverage"],
        "CE": block["coverage_error"],
        "JS": block["jensen_shannon"],
        "RCC (%)": block["rare_combination_coverage_percent"],
        "BPT": block["bits_per_token"],
        "PPL": block["perplexity"],
    })

pd.DataFrame(english_full_rows).sort_values("Dataset").reset_index(drop=True)


,Dataset,n,SV (%),Factuality,Fact coverage,CE,JS,RCC (%),BPT,PPL
0,dart,894,99.217002,0.901574,0.878776,0.000555,0.000186,25.0,0.439507,1.356141
1,webnlg,318,100.000000,0.989518,0.998249,0.000000,0.000584,100.0,0.955686,1.939502
2,zebralogic,1788,100.000000,1.000000,1.000000,0.000000,0.000066,100.0,0.862681,1.818414


In [12]:
# Tổng hợp full pipeline English.
pd.DataFrame([{
    "n": main_full["output_count"],
    "SV (%)": main_full["schema_validity_percent"],
    "Factuality": main_full["factuality_precision"],
    "Fact coverage": main_full["fact_coverage"],
    "CVR (%)": main_full["constraint_violation_rate_percent"],
    "KL": main_full["distributional_drift_kl"],
}])


,n,SV (%),Factuality,Fact coverage,CVR (%),KL
0,3000,99.766667,0.969558,0.98777,0.233333,0.010065


## 5. English baseline / ablation comparison


In [13]:
variant_names = {
    "teacher_direct": "Teacher",
    "distill_only": "Distill",
    "psdd_only": "Prior",
    "psdd_pgd": "Prior+Projection",
    "psdd_fsa": "Prior+FSA",
    "full_pipeline": "Full",
}

english_ablation = []
for key, label in variant_names.items():
    block = main_metrics["metrics"][key]
    english_ablation.append({
        "Variant": label,
        "n": block["output_count"],
        "SV (%)": block["schema_validity_percent"],
        "CVR (%)": block["constraint_violation_rate_percent"],
        "KL": block["distributional_drift_kl"],
        "Factuality": block["factuality_precision"],
    })
pd.DataFrame(english_ablation)


,Variant,n,SV (%),CVR (%),KL,Factuality
0,Teacher,100,0.000000,100.000000,0.005997,0.195000
1,Distill,100,58.000000,42.000000,0.012638,0.384909
2,Prior,100,62.000000,38.000000,0.010845,0.401757
3,Prior+Projection,100,64.000000,36.000000,0.010816,0.421757
4,Prior+FSA,300,100.000000,0.000000,0.010922,0.964444
5,Full,3000,99.766667,0.233333,0.010065,0.969558


## 6. English error analysis


In [14]:
def realized_valid(row):
    v = row["validation"]
    return bool(
        v.get("schema_valid")
        and v.get("plan_consistent")
        and v.get("required_entities_present")
        and v.get("required_relations_present")
        and v.get("fsa_accepted") is not False
    )

main_outputs = read_jsonl(require(EN_ARTIFACTS / "outputs" / "generated.jsonl"))
main_full_outputs = [r for r in main_outputs if r["variant"] == "full_pipeline"]
main_errors = [r for r in main_full_outputs if not realized_valid(r)]

print("Full-pipeline outputs:", len(main_full_outputs))
print("Residual validity errors:", len(main_errors))
print("Errors by dataset:", Counter(r["dataset"] for r in main_errors))


Full-pipeline outputs: 3000
Residual validity errors: 7
Errors by dataset: Counter({'dart': 7})


In [15]:
error_table = []
for row in main_errors:
    v = row["validation"]
    error_table.append({
        "eval_id": row["eval_id"],
        "dataset": row["dataset"],
        "text": row["text"],
        "required_entities_present": v.get("required_entities_present"),
        "required_relations_present": v.get("required_relations_present"),
        "plan_consistent": v.get("plan_consistent"),
        "fsa_accepted": v.get("fsa_accepted"),
    })
pd.DataFrame(error_table)


,eval_id,dataset,text,required_entities_present,required_relations_present,plan_consistent,fsa_accepted
0,dart:test:dart-test-1233,dart,12 TITLE The Badge and the Beautiful; [TABLECO...,True,False,True,True
1,dart:test:dart-test-1601,dart,12 TITLE The Badge and the Beautiful; [TABLECO...,True,False,True,True
2,dart:test:dart-test-1756,dart,ISO/IEC TR 15938-11 FIRST_PUBLIC_RELEASE_DATE_...,True,False,True,True
3,dart:test:dart-test-2251,dart,Souvenirs # MHR 004; Souvenirs RELEASED May 1999.,True,False,True,True
4,dart:test:dart-test-2279,dart,ISO/IEC TR 15938-8 TITLE Extraction and use of...,True,False,True,True
5,dart:test:dart-test-2514,dart,🚿 NAME SHOWER; 🚿 VALUE shower facilities.,False,True,True,True
6,dart:test:dart-test-2675,dart,71 TYPE Trailer; 71 BUILDER W. L. Holman Car C...,True,False,True,True


## 7. Vietnamese dataset: ViSFD

ViSFD là phần đánh giá tiếng Việt của đồ án. Dataset công khai được tải từ Hugging Face để kiểm tra dữ liệu gốc; experiment sử dụng 700 train, 100 validation và 200 test theo manifest đã lưu.


In [16]:
from datasets import load_dataset

visfd_ds = load_dataset("visolex/ViSFD")
visfd_raw = visfd_ds["train"]
print(visfd_raw)
print("Columns:", visfd_raw.column_names)
print("Official type counts:", Counter(str(x) for x in visfd_raw["type"]))


c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['dataset', 'type', 'index', 'comment', 'n_star', 'date_time', 'label'],
    num_rows: 11122
})
Columns: ['dataset', 'type', 'index', 'comment', 'n_star', 'date_time', 'label']
Official type counts: Counter({'train': 7786, 'test': 2224, 'dev': 1112})


In [17]:
# EDA ngắn trên dữ liệu ViSFD gốc.
raw_df = visfd_raw.to_pandas()

print("Rating distribution:")
display(raw_df["n_star"].value_counts(dropna=False).sort_index().rename_axis("n_star").reset_index(name="count"))

print("\nSample comments:")
display(raw_df[["comment", "label", "n_star", "type"]].head(5))


Rating distribution:


,n_star,count
0,1,1831
1,2,829
2,3,1426
3,4,1851
4,5,5185



Sample comments:


,comment,label,n_star,type
0,Mới mua máy này Tại thegioididong thốt nốt cảm...,{CAMERA#Positive};{FEATURES#Positive};{BATTERY...,5,train
1,Pin kém còn lại miễn chê mua 8/3/2019 tình trạ...,{BATTERY#Negative};{GENERAL#Positive};{OTHERS};,5,train
2,Sao lúc gọi điện thoại màn hình bị chấm nhỏ nh...,{FEATURES#Negative};,3,train
3,"Mọi người cập nhật phần mềm lại , nó sẽ bớt tố...",{FEATURES#Negative};{BATTERY#Neutral};{GENERAL...,3,train
4,"Mới mua Sài được 1 tháng thấy pin rất trâu, Sà...",{BATTERY#Positive};{PERFORMANCE#Positive};{SER...,5,train


## 8. ViSFD preprocessing and semantic plans

Prepared records bên dưới được tạo bởi `CircuitSynth_ViSFD/src/data.py`. Aspect--sentiment annotations được chuyển thành structured semantic plans trước khi đi qua các stage còn lại.


In [18]:
vi_manifests = read_json(require(VI_ARTIFACTS / "data" / "manifests.json"))
pd.DataFrame(vi_manifests)[
    ["dataset", "official_split", "configured_population_count", "selected_count", "duplicate_count", "leakage_count"]
]


,dataset,official_split,configured_population_count,selected_count,duplicate_count,leakage_count
0,visfd,test,2224,200,0,0
1,visfd,train,7786,700,0,0
2,visfd,validation,1112,100,0,0


In [19]:
vi_records = read_jsonl(require(VI_ARTIFACTS / "data" / "records.jsonl"))

print("Prepared ViSFD records:", len(vi_records))
print("Split counts:", Counter(r["official_split"] for r in vi_records))

pd.DataFrame([
    {
        "source_id": row["source_id"],
        "split": row["official_split"],
        "reference_plan": row["reference_plan"],
        "reference_text": row.get("reference_text", ""),
    }
    for row in vi_records[:5]
])


Prepared ViSFD records: 1000
Split counts: Counter({'train': 700, 'test': 200, 'validation': 100})


,source_id,split,reference_plan,reference_text
0,visfd:test:10000,test,"{'domain': 'visfd', 'others': False, 'pairs': ...","Pin khỏe, máy mượt, bút nhạy, nghe nhạc cực ha..."
1,visfd:test:10001,test,"{'domain': 'visfd', 'others': False, 'pairs': ...",Máy sau khi cập nhật thì lỗi định vị và hào p...
2,visfd:test:10010,test,"{'domain': 'visfd', 'others': False, 'pairs': ...","Máy cũng hơi OK mà pin mau tuốt, cấu hình tạm ..."
3,visfd:test:10037,test,"{'domain': 'visfd', 'others': False, 'pairs': ...","Đã mua đc 2 tháng, pin ở tầm trung sài đc hơn ..."
4,visfd:test:10038,test,"{'domain': 'visfd', 'others': False, 'pairs': ...",Pin trâu dùng cả ngày ko hết chơi liên quân ph...


## 9. ViSFD pipeline and Student training


In [20]:
vi_config = yaml.safe_load((VISFD_ROOT / "config.yaml").read_text(encoding="utf-8"))
vi_train = read_json(require(VI_ARTIFACTS / "student" / "training_report.json"))
vi_quota = read_json(require(VI_ARTIFACTS / "silver" / "quota_report.json"))
vi_projection = read_json(require(VI_ARTIFACTS / "circuits" / "projection_report.json"))

pd.DataFrame([{
    "Teacher": vi_config["teacher"]["model"],
    "Student": vi_config["student"]["model"],
    "Silver accepted": vi_quota["accepted_total"],
    "Semantic plan source": vi_quota.get("semantic_plan_sources", {}),
    "Student train": vi_train["train_count"],
    "Student validation": vi_train["validation_count"],
    "Optimizer steps": vi_train["executed_optimizer_steps"],
    "Epochs": vi_config["student"]["epochs"],
    "QLoRA r": vi_config["student"]["lora_rank"],
    "Gradient accumulation": vi_config["student"]["gradient_accumulation_steps"],
}])


,Teacher,Student,Silver accepted,Semantic plan source,Student train,Student validation,Optimizer steps,Epochs,QLoRA r,Gradient accumulation
0,Qwen/Qwen2.5-1.5B-Instruct,Qwen/Qwen2.5-0.5B-Instruct,700,{'human_annotation': 700},665,35,84,1,8,8


In [21]:
pd.DataFrame([{
    "Projection converged": vi_projection["objective_converged"],
    "Iterations": vi_projection["iterations"],
    "Optimality residual": vi_projection["optimality_residual"],
    "Invalid support mass": vi_projection["invalid_support_mass"],
    "Normalized": vi_projection["normalized"],
    "Solver": vi_projection["solver"],
}])


,Projection converged,Iterations,Optimality residual,Invalid support mass,Normalized,Solver
0,True,3,3.872925e-08,0.0,True,convex_dual_lbfgs


## 10. ViSFD test evaluation


In [22]:
vi_metrics = read_json(require(VI_ARTIFACTS / "metrics" / "metrics.json"))
vi_full = vi_metrics["metrics"]["full_pipeline"]

pd.DataFrame([{
    "n": vi_full["output_count"],
    "SV (%)": vi_full["schema_validity_percent"],
    "Factuality": vi_full["factuality_precision"],
    "Fact coverage": vi_full["fact_coverage"],
    "CE": vi_full["coverage_error"],
    "JS": vi_full["jensen_shannon"],
    "RCC (%)": vi_full["rare_combination_coverage_percent"],
    "BPT": vi_full["bits_per_token"],
    "PPL": vi_full["perplexity"],
}])


,n,SV (%),Factuality,Fact coverage,CE,JS,RCC (%),BPT,PPL
0,200,100.0,0.720161,1.0,0.007951,0.001083,86.956522,1.012471,2.017363


## 11. ViSFD baseline / ablation comparison


In [23]:
vi_ablation = []
for key, label in variant_names.items():
    block = vi_metrics["metrics"][key]
    vi_ablation.append({
        "Variant": label,
        "n": block["output_count"],
        "SV (%)": block["schema_validity_percent"],
        "CVR (%)": block["constraint_violation_rate_percent"],
        "KL": block["distributional_drift_kl"],
        "Factuality": block["factuality_precision"],
    })
pd.DataFrame(vi_ablation)


,Variant,n,SV (%),CVR (%),KL,Factuality
0,Teacher,20,20.0,80.0,0.120461,0.161667
1,Distill,20,5.0,95.0,0.132314,0.258333
2,Prior,20,0.0,100.0,0.165466,0.196250
3,Prior+Projection,20,0.0,100.0,0.226184,0.191667
4,Prior+FSA,50,100.0,0.0,0.012412,0.703929
5,Full,200,100.0,0.0,0.002983,0.720161


In [24]:
# Kết quả full pipeline theo nhóm rating.
star_rows = []
for group, block in sorted(vi_full["by_group"].items()):
    star_rows.append({
        "Group": group,
        "n": block["output_count"],
        "SV (%)": block["schema_validity_percent"],
        "Factuality": block["factuality_precision"],
        "RCC (%)": block["rare_combination_coverage_percent"],
    })
pd.DataFrame(star_rows)


,Group,n,SV (%),Factuality,RCC (%)
0,star_1,34,100.0,0.719328,70.588235
1,star_2,17,100.0,0.652941,50.000000
2,star_3,26,100.0,0.629487,70.000000
3,star_4,30,100.0,0.681521,45.000000
4,star_5,93,100.0,0.770567,78.947368


## 12. ViSFD error analysis


In [25]:
vi_outputs = read_jsonl(require(VI_ARTIFACTS / "outputs" / "generated.jsonl"))
vi_full_outputs = [r for r in vi_outputs if r["variant"] == "full_pipeline"]
vi_errors = [r for r in vi_full_outputs if not realized_valid(r)]

print("Full-pipeline ViSFD outputs:", len(vi_full_outputs))
print("Residual validity errors:", len(vi_errors))

# Một vài outputs để kiểm tra trực tiếp plan -> Vietnamese realization.
display(pd.DataFrame([
    {
        "eval_id": row["eval_id"],
        "group": row["evaluation_group"],
        "plan": row["plan"],
        "text": row["text"],
        "schema_valid": row["validation"]["schema_valid"],
        "fsa_accepted": row["validation"]["fsa_accepted"],
    }
    for row in vi_full_outputs[:10]
]))


Full-pipeline ViSFD outputs: 200
Residual validity errors: 0


,eval_id,group,plan,text,schema_valid,fsa_accepted
0,visfd:test:10000,star_1,"{'domain': 'visfd', 'others': False, 'pairs': ...",tính năng có đánh giá tiêu cực; tổng thể có đá...,True,True
1,visfd:test:10001,star_5,"{'domain': 'visfd', 'others': False, 'pairs': ...",dịch vụ và phụ kiện có đánh giá tích cực; hiệu...,True,True
2,visfd:test:10010,star_1,"{'domain': 'visfd', 'others': False, 'pairs': ...",pin có đánh giá tiêu cực; tính năng có đánh gi...,True,True
3,visfd:test:10037,star_2,"{'domain': 'visfd', 'others': True, 'pairs': [...",camera có đánh giá tích cực; tổng thể có đánh ...,True,True
4,visfd:test:10038,star_4,"{'domain': 'visfd', 'others': True, 'pairs': [...",tổng thể có đánh giá tích cực; tính năng có đá...,True,True
5,visfd:test:10054,star_4,"{'domain': 'visfd', 'others': False, 'pairs': ...",tính năng có đánh giá tiêu cực; thiết kế có đá...,True,True
6,visfd:test:10072,star_5,"{'domain': 'visfd', 'others': True, 'pairs': [...",tổng thể có đánh giá tích cực; dịch vụ và phụ ...,True,True
7,visfd:test:10086,star_5,"{'domain': 'visfd', 'others': True, 'pairs': [...",tổng thể có đánh giá tích cực; hiệu năng có đá...,True,True
8,visfd:test:10128,star_4,"{'domain': 'visfd', 'others': False, 'pairs': ...",tổng thể có đánh giá tích cực; pin có đánh giá...,True,True
9,visfd:test:10151,star_5,"{'domain': 'visfd', 'others': False, 'pairs': ...",pin có đánh giá tích cực; camera có đánh giá t...,True,True


## 13. Demo trên semantic plan tiếng Việt mới

Demo nhẹ dùng chính `controlled_realizations` trong FSA module của nhánh ViSFD. Cell tiếp theo có thể nạp Student QLoRA thật khi GPU và adapter đều sẵn sàng.


In [30]:
new_plan = {
    "domain": "visfd",
    "pairs": [
        {"aspect": "CAMERA", "sentiment": "Positive"},
        {"aspect": "BATTERY", "sentiment": "Negative"},
    ],
    "others": False,
}

plan_json = json.dumps(new_plan, ensure_ascii=False)
demo_code = f'''
import json
from src.fsa import controlled_realizations
plan = json.loads({plan_json!r})
print("Plan:", json.dumps(plan, ensure_ascii=False, indent=2))
print("Controlled Vietnamese realizations:")
for text in controlled_realizations(plan):
    print("-", text)
'''
run_cmd([sys.executable, "-c", demo_code], VISFD_ROOT)


Plan: {
  "domain": "visfd",
  "pairs": [
    {
      "aspect": "CAMERA",
      "sentiment": "Positive"
    },
    {
      "aspect": "BATTERY",
      "sentiment": "Negative"
    }
  ],
  "others": false
}
Controlled Vietnamese realizations:
- camera có đánh giá tích cực; pin có đánh giá tiêu cực.
- pin có đánh giá tiêu cực; camera có đánh giá tích cực.



CompletedProcess(args=['c:\\Users\\ASUS\\AppData\\Local\\Programs\\Python\\Python311\\python.exe', '-c', '\nimport json\nfrom src.fsa import controlled_realizations\nplan = json.loads(\'{"domain": "visfd", "pairs": [{"aspect": "CAMERA", "sentiment": "Positive"}, {"aspect": "BATTERY", "sentiment": "Negative"}], "others": false}\')\nprint("Plan:", json.dumps(plan, ensure_ascii=False, indent=2))\nprint("Controlled Vietnamese realizations:")\nfor text in controlled_realizations(plan):\n    print("-", text)\n'], returncode=0, stdout='Plan: {\n  "domain": "visfd",\n  "pairs": [\n    {\n      "aspect": "CAMERA",\n      "sentiment": "Positive"\n    },\n    {\n      "aspect": "BATTERY",\n      "sentiment": "Negative"\n    }\n  ],\n  "others": false\n}\nControlled Vietnamese realizations:\n- camera có đánh giá tích cực; pin có đánh giá tiêu cực.\n- pin có đánh giá tiêu cực; camera có đánh giá tích cực.\n')

In [27]:
# Neural demo: tự chạy khi có CUDA và ViSFD adapter thật trong artifacts/student/final/.
adapter_path = VI_ARTIFACTS / "student" / "final" / "adapter_model.safetensors"
RUN_NEURAL_DEMO = bool(torch.cuda.is_available() and adapter_path.exists())
print("RUN_NEURAL_DEMO:", RUN_NEURAL_DEMO)

if RUN_NEURAL_DEMO:
    plan_json = json.dumps(new_plan, ensure_ascii=False)
    neural_code = f'''
import json
from pathlib import Path
import yaml
from src.student import load_full_student, build_student_prompt, render_generation_prompt
from src.fsa import build_plan_fsa, constrained_full_generate

root = Path.cwd()
config = yaml.safe_load((root / "config.yaml").read_text(encoding="utf-8"))
plan = json.loads({plan_json!r})
model, tokenizer = load_full_student(config, root / "artifacts" / "student")
prompt = render_generation_prompt(tokenizer, build_student_prompt(plan))
fsa = build_plan_fsa(plan, tokenizer)
text, tokens, logps = constrained_full_generate(
    model, tokenizer, prompt, fsa,
    int(config["generation"]["max_new_tokens"]),
    int(config["project"]["seed"]) + 9000,
)
print("Generated Vietnamese text:", text)
print("Generated tokens:", len(tokens))
'''
    run_cmd([sys.executable, "-c", neural_code], VISFD_ROOT)
else:
    print("Neural demo skipped because CUDA or the trained ViSFD adapter is unavailable. The FSA demo above remains executable on CPU.")


RUN_NEURAL_DEMO: False
Neural demo skipped because CUDA or the trained ViSFD adapter is unavailable. The FSA demo above remains executable on CPU.


## 14. Optional full rerun

Artifacts ở các phần trên là kết quả của những lần chạy hoàn chỉnh đã lưu trong repository. Để tái chạy toàn bộ experiment từ Stage 1 đến Stage 8, bật các cờ bên dưới. Mặc định notebook không tự train lại hai mô hình khi bấm **Run all**, nhằm giữ thời gian chạy hợp lý trên Colab/Kaggle.


In [28]:
RERUN_MAIN_PIPELINE = False
RERUN_VISFD_PIPELINE = False

if RERUN_MAIN_PIPELINE:
    run_cmd([sys.executable, "run.py", "run"], MAIN_ROOT)
else:
    print("Main pipeline rerun: skipped; submitted artifacts are used.")

if RERUN_VISFD_PIPELINE:
    run_cmd([sys.executable, "run.py", "run"], VISFD_ROOT)
else:
    print("ViSFD pipeline rerun: skipped; submitted artifacts are used.")


Main pipeline rerun: skipped; submitted artifacts are used.
ViSFD pipeline rerun: skipped; submitted artifacts are used.


## 15. Summary

Notebook đã kiểm tra source và test của hai pipeline, đọc prepared datasets và artifacts, trình bày cấu hình huấn luyện, đánh giá English benchmarks, so sánh các ablation, phân tích residual errors, tải và khảo sát ViSFD, đánh giá phần tiếng Việt và demo một semantic plan tiếng Việt mới.
